# Traveler Insights · Pipeline CRISP-ML(Q) paso a paso
**Cómo se entrena, evalúa, exporta e integra el modelo en la app desplegada en Vercel**

Este notebook ejecuta, celda por celda, lo mismo que hace el comando
`python -m traveler_insights.pipeline`. La diferencia es que aquí se puede ver y discutir cada decisión. Usa los módulos del repositorio, así que lo que se aprende aquí es exactamente lo que corre en GitHub Actions y en producción.

| Fase CRISP-ML(Q) | Sección | Módulo del repositorio |
|---|---|---|
| 1. Negocio y datos | §1 | `docs/crisp-mlq/01_entendimiento.md`, `configs/params.yaml` |
| 2. Preparación de datos | §2 | `ml/data.py` |
| 3. Modelado | §3 | `ml/train.py` |
| 4. Evaluación | §4 y §5 | `ml/evaluate.py` |
| 5. Despliegue | §6 y §7 | `ml/export.py`, `core/`, `app.py` |
| 6. Monitoreo | §8 | `.github/workflows/monitor.yml` |

**Requisitos:**
- Abrir el notebook desde la carpeta `notebooks/` del repositorio.
- Haber instalado las dependencias con `pip install -e ".[train,dev]"`.
- Tener `data/raw/train.csv` (y opcionalmente `test.csv`) de Kaggle. Ver `data/README.md`.

## 0. Configuración

In [ ]:
import json, random, sys
from pathlib import Path

# Raíz del repositorio (el notebook vive en notebooks/)
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

from traveler_insights.ml.data import preparar
from traveler_insights.ml.train import baselines, modelo_candidato
from traveler_insights.ml import evaluate as ev
from traveler_insights.ml.export import a_json, paridad, guardar
from traveler_insights.core.model import LinearTextModel
from traveler_insights.core.business import analizar, mapa_polaridad

CFG = yaml.safe_load((ROOT / "configs" / "params.yaml").read_text(encoding="utf-8"))
TRAIN_CSV = ROOT / CFG["data"]["train_path"]     # data/raw/train.csv
TEST_CSV  = TRAIN_CSV.with_name("test.csv")       # opcional: test sin etiquetas de Kaggle
SEED = CFG["seed"]
GUARDAR_MODELO = False   # True = sobrescribe model/ (equivale a desplegar en el próximo push)

pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.dpi"] = 110
ROJO, GRIS, OSCURO = "#C10230", "#6F7271", "#222222"
print("Repositorio:", ROOT)
print("Datos:", TRAIN_CSV, "| existe:", TRAIN_CSV.exists())

## 1. Fase 1 · Entendimiento del negocio
Antes de tocar los datos se fija **qué es un buen modelo**. Estos umbrales viven en `configs/params.yaml` y no se cambian después de ver los resultados: hacerlo sería ajustar la meta al disparo.

In [ ]:
pd.DataFrame([
    ["Macro-F1 mínimo en prueba", CFG["quality_gate"]["min_macro_f1"]],
    ["Mejora mínima sobre la clase mayoritaria", CFG["quality_gate"]["must_beat_dummy_by"]],
    ["Regresión máxima frente al modelo en producción", CFG["quality_gate"]["max_regression_vs_production"]],
    ["Error máximo de paridad sklearn ↔ JSON", CFG["quality_gate"]["max_parity_error"]],
    ["Umbral de confianza τ para automatizar", CFG["serving"]["tau"]],
], columns=["Criterio de aceptación", "Valor"])

## 2. Fase 2 · Preparación y validación de datos
`preparar()` limpia los datos y verifica los requisitos de calidad. Si algún control falla, el pipeline real se detiene aquí.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
datos, calidad = preparar(df_raw, CFG["data"])
print(f"Columnas detectadas → texto: {calidad['columns']['text']} | etiqueta: {calidad['columns']['label']}")
print(f"Filas: {calidad['rows_raw']} originales → {calidad['rows_final']} finales "
      f"| duplicados {calidad['duplicate_rate']:.1%} | conflictos {calidad['conflict_rate']:.1%}")
pd.DataFrame([{"control": k, "aprobado": "✔" if v["passed"] else "✘", "detalle": v["detail"]}
              for k, v in calidad["checks"].items()])

In [ ]:
dist = pd.Series(calidad["class_distribution"]).sort_values()
ax = dist.plot.barh(color=ROJO, figsize=(7, 2.8))
ax.set_title(f"Distribución de clases · razón de desbalance {calidad['imbalance_ratio']}")
ax.set_xlabel("reseñas"); plt.tight_layout(); plt.show()
assert calidad["passed"], "Los datos no cumplen los requisitos: revise configs/params.yaml › data.quality"

## 3. Fase 3 · Modelado: baselines y candidato
- **Protocolo:** la prueba se separa *antes* de todo y se usa una sola vez. Los modelos se comparan con validación cruzada estratificada, con las mismas particiones y la misma semilla.
- **Modelos:**
  - clase mayoritaria: el piso mínimo;
  - TF-IDF + Naive Bayes: el clásico;
  - candidato: TF-IDF de palabras y caracteres + regresión logística.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(datos["texto"], datos["etiqueta"], test_size=CFG["data"]["test_size"],
                                          stratify=datos["etiqueta"], random_state=SEED)
print(f"Entrenamiento: {len(X_tr)} | Prueba (se toca una vez): {len(X_te)}")

modelos = {**baselines(CFG["model"]), "candidato": modelo_candidato(CFG["model"])}
filas, preds = [], {}
for nombre, m in modelos.items():
    cv = ev.validacion_cruzada(m, X_tr, y_tr, CFG["evaluation"]["cv_folds"], SEED)
    m.fit(X_tr, y_tr)
    preds[nombre] = m.predict(X_te)
    filas.append({"modelo": nombre, "CV macro-F1": f"{cv['macro_f1_mean']:.4f} ± {cv['macro_f1_std']:.4f}",
                  "prueba macro-F1": ev.metricas_prueba(y_te, preds[nombre], m.classes_)["macro_f1"]})
pd.DataFrame(filas).set_index("modelo")

**Para discutir:** si la desviación entre pliegues es mayor que la diferencia entre dos modelos, todavía no se puede afirmar que uno sea mejor.

## 4. Fase 4 · Evaluación rigurosa del candidato

In [ ]:
cand = modelos["candidato"]
proba = cand.predict_proba(X_te)
met = ev.metricas_prueba(y_te, preds["candidato"], cand.classes_)
print(f"Exactitud {met['accuracy']} | macro-F1 {met['macro_f1']}")
display(pd.DataFrame(met["per_class"]).T.round(3))
ConfusionMatrixDisplay.from_predictions(y_te, preds["candidato"], cmap="Reds", xticks_rotation=45)
plt.title("Matriz de confusión · candidato"); plt.tight_layout(); plt.show()

### 4.1 ¿Qué tan seguro es el número? Intervalo por bootstrap
Se remuestrea el conjunto de prueba muchas veces y se observa cuánto varía el macro-F1.

In [ ]:
B = CFG["evaluation"]["bootstrap_samples"]
ic = ev.bootstrap_ic(y_te, preds["candidato"], B, SEED)
rng = np.random.default_rng(SEED); yt, yp = np.asarray(y_te), np.asarray(preds["candidato"])
from sklearn.metrics import f1_score
muestras = [f1_score(yt[i], yp[i], average="macro") for i in (rng.integers(0, len(yt), len(yt)) for _ in range(B))]
plt.figure(figsize=(7, 2.8)); plt.hist(muestras, bins=40, color=GRIS)
for v in (ic["low"], ic["high"]): plt.axvline(v, color=ROJO, ls="--")
plt.title(f"Bootstrap del macro-F1 · IC 95 % [{ic['low']}, {ic['high']}]"); plt.tight_layout(); plt.show()

### 4.2 ¿La mejora sobre el baseline es real? Prueba de McNemar
Compara **los mismos ejemplos**: cuántos acierta solo el candidato frente a cuántos acierta solo Naive Bayes (Dietterich, 1998).

In [ ]:
mc = ev.mcnemar(y_te, preds["candidato"], preds["tfidf_naive_bayes"])
print(f"Solo el candidato acierta: {mc['only_candidate_correct']} | solo Naive Bayes acierta: {mc['only_baseline_correct']} | p = {mc['p_value']}")
print("→ Diferencia significativa al 5 %" if mc["p_value"] < 0.05 else "→ No hay evidencia suficiente de diferencia")

### 4.3 ¿La confianza significa algo? Calibración
Como el umbral τ decide qué se automatiza, la confianza debe corresponder a la exactitud real (Guo et al., 2017).

In [ ]:
cal = ev.calibracion(y_te, proba, cand.classes_, CFG["evaluation"]["calibration_bins"])
c = pd.DataFrame(cal["reliability"])
plt.figure(figsize=(4.2, 4.2))
plt.plot([0, 1], [0, 1], ls="--", color=GRIS, label="calibración perfecta")
plt.plot(c.confidence, c.accuracy, "o-", color=ROJO, label="modelo")
plt.xlabel("confianza predicha"); plt.ylabel("exactitud real"); plt.legend()
plt.title(f"Curva de fiabilidad · ECE = {cal['ece']}"); plt.tight_layout(); plt.show()

### 4.4 ¿Falla en algún grupo? Desempeño por longitud de reseña

In [ ]:
pd.DataFrame(ev.por_subgrupo(X_te, y_te, preds["candidato"])).T

### 4.5 ¿Cuánto se puede automatizar? Curva del umbral τ

In [ ]:
tc = pd.DataFrame(ev.curva_tau(y_te, proba, cand.classes_))
plt.figure(figsize=(7, 3))
plt.plot(tc.tau, tc.coverage, "o-", color=ROJO, label="% automatizado")
plt.plot(tc.tau, tc.accuracy_auto, "s-", color=OSCURO, label="exactitud en lo automatizado")
plt.axvline(CFG["serving"]["tau"], color=GRIS, ls="--"); plt.ylim(0, 1.02); plt.legend(); plt.xlabel("τ")
plt.title("Compromiso cobertura vs. calidad"); plt.tight_layout(); plt.show()
tc

### 4.6 Compuerta de calidad

In [ ]:
res = {"candidate": {"test": met}, "baselines": {"dummy_mayoritaria": {"test": ev.metricas_prueba(y_te, preds["dummy_mayoritaria"], modelos["dummy_mayoritaria"].classes_)}}}
prod_path = ROOT / "model" / "metrics.json"
produccion = json.loads(prod_path.read_text(encoding="utf-8")) if prod_path.exists() else None
gate = ev.compuerta(res, CFG["quality_gate"], produccion)
print("Compuerta:", "APROBADO ✔" if gate["passed"] else "RECHAZADO ✘", "| comparado con producción:", produccion is not None)
pd.DataFrame([{"control": k, "aprobado": "✔" if v["passed"] else "✘", "detalle": v["detail"]} for k, v in gate["checks"].items()])

## 5. Pruebas de comportamiento sobre reseñas reales
Inspiradas en CheckList (Ribeiro et al., 2020), estas pruebas no inventan reseñas: **alteran reseñas reales de la prueba** y miden cuántas veces cambia la predicción. Un modelo robusto debería cambiar poco.
- **Errores tipográficos:** se intercambian dos letras vecinas en una palabra.
- **Mayúsculas y tildes:** el texto pasa a MAYÚSCULAS y se le quitan las tildes.
- **Frase irrelevante:** se agrega "Viajamos en familia." al final.

In [ ]:
import unicodedata
rnd = random.Random(SEED)
def typo(t):
    ws = t.split(); idx = [i for i, w in enumerate(ws) if len(w) > 3]
    if not idx: return t
    i = rnd.choice(idx); w = ws[i]; j = rnd.randrange(len(w) - 1)
    ws[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]; return " ".join(ws)
def mayus_sin_tildes(t):
    return "".join(ch for ch in unicodedata.normalize("NFKD", t.upper()) if not unicodedata.combining(ch))
def frase_irrelevante(t): return t.rstrip() + " Viajamos en familia."

muestra = X_te.sample(min(500, len(X_te)), random_state=SEED)
base = cand.predict(muestra)
filas = []
for nombre, f in [("errores tipográficos", typo), ("mayúsculas y tildes", mayus_sin_tildes), ("frase irrelevante", frase_irrelevante)]:
    pert = cand.predict(muestra.map(f))
    filas.append({"prueba": nombre, "% predicciones que cambian": round(100 * float((pert != base).mean()), 2)})
pd.DataFrame(filas)

**Para discutir:** ¿por qué "mayúsculas y tildes" debería dar casi 0 %? Mire `normalizar()` en `core/text.py`. Si los errores tipográficos cambian muchas predicciones, ¿ayudan los n-gramas de caracteres?

## 6. Fase 5 · Exportación para producción
scikit-learn es demasiado pesado para una función de Vercel. Por eso el modelo **final**, reentrenado con todos los datos, se exporta a JSON: vocabulario, IDF y pesos. La inferencia se hace en Python puro (`core/model.py`), y antes de aceptar la exportación se verifica que ambos den las mismas probabilidades.

In [ ]:
final = modelo_candidato(CFG["model"]).fit(datos["texto"], datos["etiqueta"])
meta = {"columns": calidad["columns"], "polarity": mapa_polaridad([str(c) for c in final.classes_]),
        "tau": CFG["serving"]["tau"], "test_macro_f1": met["macro_f1"]}
spec = a_json(final, meta)
err = paridad(final, spec, X_te.head(300))
tam = len(json.dumps(spec, separators=(",", ":")).encode()) / 1e6
print(f"Paridad sklearn ↔ JSON: error máx. {err:.2e} (límite {CFG['quality_gate']['max_parity_error']})")
print(f"Tamaño del modelo JSON: {tam:.1f} MB | términos por bloque:", {b['name']: len(b['features']) for b in spec["blocks"]})
print("Polaridad deducida por clase (revísela):", meta["polarity"])

In [ ]:
if GUARDAR_MODELO and gate["passed"] and err <= CFG["quality_gate"]["max_parity_error"]:
    guardar(spec, ROOT / "model" / "model.json")
    print("Modelo guardado en model/model.json. Para desplegarlo: commit + Pull Request + merge.")
    print("Recomendado: usar `python -m traveler_insights.pipeline`, que además escribe metrics.json y MODEL_CARD.md.")
else:
    print("No se guardó el modelo (GUARDAR_MODELO = False o la compuerta no aprobó).")

## 7. Integración: del modelo a la app
### 7.1 La lógica de negocio sobre el modelo exportado
`analizar()` es exactamente la función que llama la API: clase, confianza, aspectos, respuesta sugerida y decisión (automatizar o revisión humana).

In [ ]:
modelo_js = LinearTextModel(spec)
pd.DataFrame(analizar(modelo_js, list(X_te.head(6)), tau=CFG["serving"]["tau"]))[
    ["texto", "clase", "confianza", "aspectos", "decision", "motivos", "respuesta_sugerida"]]

### 7.2 La API, probada sin servidor
Se usa el mismo `app.py` que despliega Vercel, apuntándolo al modelo de este notebook.

In [ ]:
import tempfile
from fastapi.testclient import TestClient
import app as api

tmp = Path(tempfile.mkdtemp()) / "model.json"; guardar(spec, tmp)
api.MODEL_PATH = tmp; api.modelo.cache_clear()
cliente = TestClient(api.app)
print(cliente.get("/api/health").json())
r = cliente.post("/api/analyze", json={"texts": list(X_te.head(2)), "tau": CFG["serving"]["tau"]})
print(json.dumps(r.json(), ensure_ascii=False, indent=2)[:1200])

### 7.3 La app desplegada en Vercel
Cuando la app esté publicada, escriba su URL en `APP_URL` para consultarla desde aquí, como lo haría cualquier sistema de la organización.

In [ ]:
APP_URL = ""   # p. ej. "https://traveler-insights.vercel.app"
if APP_URL:
    import urllib.request
    req = urllib.request.Request(f"{APP_URL}/api/analyze", method="POST",
                                 data=json.dumps({"texts": list(X_te.head(3))}).encode(),
                                 headers={"Content-Type": "application/json"})
    print(json.dumps(json.load(urllib.request.urlopen(req)), ensure_ascii=False, indent=2)[:1500])
else:
    print("Defina APP_URL para consultar la app desplegada.")

## 8. Fase 6 · Monitoreo: ¿los datos nuevos se parecen a los de entrenamiento?
El `test.csv` de Kaggle no trae etiquetas, igual que los datos reales de producción. Aun así se puede vigilar la **deriva**: si la distribución de clases predichas o de confianza cambia mucho respecto al entrenamiento, el modelo puede estar fuera de su zona de validez.

Se usa el **índice de estabilidad poblacional (PSI)**. Como regla práctica: PSI < 0,1 indica estabilidad; entre 0,1 y 0,25, un cambio moderado; mayor que 0,25, un cambio importante que conviene revisar.

In [ ]:
def psi(ref, act, eps=1e-6):
    ref, act = np.asarray(ref) + eps, np.asarray(act) + eps
    ref, act = ref / ref.sum(), act / act.sum()
    return float(np.sum((act - ref) * np.log(act / ref)))

if TEST_CSV.exists():
    df_new = pd.read_csv(TEST_CSV)
    col = calidad["columns"]["text"] if calidad["columns"]["text"] in df_new.columns else \
          max([c for c in df_new.columns if pd.api.types.is_string_dtype(df_new[c]) or pd.api.types.is_object_dtype(df_new[c])], key=lambda c: df_new[c].astype(str).str.len().mean())
    nuevos = df_new[col].fillna("").astype(str)
    clases = list(final.classes_)
    p_ref = pd.Series(final.predict(X_te)).value_counts().reindex(clases, fill_value=0)
    p_new = pd.Series(final.predict(nuevos)).value_counts().reindex(clases, fill_value=0)
    bins = np.linspace(0, 1, 11)
    c_ref = np.histogram(final.predict_proba(X_te).max(1), bins)[0]
    c_new = np.histogram(final.predict_proba(nuevos).max(1), bins)[0]
    print(f"PSI clases predichas: {psi(p_ref, p_new):.3f} | PSI confianza: {psi(c_ref, c_new):.3f}")
    display(pd.DataFrame({"referencia (prueba)": (p_ref / p_ref.sum()).round(3), "nuevos (test.csv)": (p_new / p_new.sum()).round(3)}))
else:
    print("No se encontró test.csv; descárguelo de Kaggle para ejecutar esta sección.")

## 9. Reflexión crítica (entregable)
1. ¿El candidato supera a Naive Bayes de forma **significativa** (McNemar) y **relevante** para el negocio?
2. ¿El intervalo de confianza del macro-F1 queda por encima del umbral de la compuerta, o solo el valor puntual?
3. ¿El modelo está calibrado? ¿Qué τ recomendaría y por qué?
4. ¿Qué prueba de comportamiento le preocupa más? Proponga una mejora.
5. ¿Qué indicaría un PSI alto en producción y qué haría el equipo?
6. Si reemplazara este modelo por su Transformer afinado, ¿qué partes del repositorio cambiarían y cuáles no? (README, sección 10)

### Referencias
- Studer, S. et al. (2021). Towards CRISP-ML(Q). *Machine Learning and Knowledge Extraction*, 3(2), 392–413.
- Dietterich, T. G. (1998). Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms. *Neural Computation*, 10(7).
- Guo, C. et al. (2017). On Calibration of Modern Neural Networks. *ICML 2017*.
- Ribeiro, M. T. et al. (2020). Beyond Accuracy: Behavioral Testing of NLP Models with CheckList. *ACL 2020*.